# Numerical Computation of The Bayes Update Rule for Bayesian Adaptive Filtering

This notebook aims to implement through numerical integration the unidimensional bayesian update rule for general distributions in both the prior and the likelihood.

$$ f_m(\theta_{t,m} | y_{1:t}) \propto f_m(\theta_{t,m} | y_{1:t-1})\,f_{\zeta_m}\!\big(y_t - x_{t,m}\theta_{t,m}\big) $$

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    NLMS_parameters, sKF_parameters, sKF_L_parameters, sKF_int_parameters, sKF_L_int_parameters,
    std_env_parameters, laplace_env_parameters,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, std_behavior,
    # algorithms
    NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations_Modular_Variance, MC_Simulations,
    std_gaussian_behavior, laplace_noise_behavior,
)

## Framework

All filter implementations, parameter dtypes and Monte Carlo drivers live in [`filters.py`](filters.py).

## Simulations

### Monte Carlo Simulations

#### Bayesian Numerical Filter

##### Gaussian Likelihood Simulations

In [ ]:
L = 3
ho = np.sinc(np.linspace(0,1.5,L))
ho = ho/np.linalg.norm(ho) # ground truth
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
#AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx = autocorr_matrix_calc(AR, 1, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig)/np.min(eig)
print(f'Eigenvalue spread: {chi:.4} \n')

gaussian_env_setup = std_env_parameters(ho=ho, AR=AR, var_v=var_v, var_x=var_x)

##### Bayesian Likelihood Simulations


In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2 # v_tilde_0
var_eta = 10*var_v
dx_factor = 1/10
min_std_deviations = 5

Algorithms = [sKF_integral_algorithm, sKF_algorithm]

skf_int_params = sKF_int_parameters("sKF_integral", epsilon, var_eta, var_theta_0, dx_factor, min_std_deviations)
sKF_parameters_paper = sKF_parameters("sKF_paper", epsilon, var_eta, var_theta_0)

Alg_Parameters = [skf_int_params, sKF_parameters_paper]

with ProgressBar(total=NR) as PBar:
  MC_measures = MC_Simulations(N, 
                                NR, 
                                gaussian_env_setup,
                                std_gaussian_behavior,
                                Algorithms,
                                Alg_Parameters,
                                h0,
                                PBar = PBar)

###### Graphic Results

In [ ]:
plt.plot(10*np.log10(MC_measures["sKF_integral"]['MSD']))
plt.plot(10*np.log10(MC_measures["sKF_paper"]['MSD']))
plt.ylabel("MSD (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
#h_L_num  = MC_measures["sKF_L_paper"]['h']
#h_L_50_51   = MC_measures["sKF_L_paper"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  #plt.plot(h_L_50_51[:, k], color='k', linestyle='-.', label=f"$w_{k}$ paper Laplace")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(10*np.log10(np.abs(h_num[:, k] - h_paper[:,k])/np.abs(h_paper[:,k])),
           color=c, linestyle='-',
           label=f"$w_{k}$ numerical")

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("weights difference (numerical-paper)/paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))

plt.plot(v_num[:, 0],   color=c, linestyle='-',  label=f"v numerical")
plt.plot(v_paper[:, 0], color=c, linestyle='--', label=f"v paper")

#plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("v")
plt.xlabel("Iterations")
plt.title("sKF variance: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))

plt.plot(10*np.log10(np.abs(v_num[:, 0] - v_paper[:, 0])/np.abs(v_paper[:, 0])), color=c, linestyle='--', label=f"v paper")

plt.ylabel("(v_num-v_paper)/v_paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF variance difference: numerical integration vs paper recursion LOG")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

##### Laplacian Likelihood

In [ ]:
L = 3
ho = np.sinc(np.linspace(0, 1.5, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
scale_v = np.sqrt(1e-3)

# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
# AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

laplace_env_setup = laplace_env_parameters(ho=ho, AR=AR, scale_v=scale_v, var_x=var_x)

Rxx = autocorr_matrix_calc(AR, 1, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig) / np.min(eig)
print(f"Eigenvalue spread: {chi:.4} \n")

In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2  # v_tilde_0
b_eta = 5 * np.sqrt(var_v)
dx_factor = 1/10
min_std_deviations = 3

Algorithms = [sKF_L_integral_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm]

skf_L_int_params = sKF_L_int_parameters("sKF_L_integral", epsilon, b_eta, var_theta_0, dx_factor, min_std_deviations)
sKF_L_minorized_params = sKF_L_parameters("sKF_L_minorized", epsilon, b_eta, var_theta_0)
sKF_L_exact_params = sKF_L_parameters("sKF_L_exact", epsilon, b_eta, var_theta_0)

Alg_Parameters = [skf_L_int_params, sKF_L_minorized_params, sKF_L_exact_params]

with ProgressBar(total=NR) as PBar:
    MC_measures = MC_Simulations(
        N, NR, laplace_env_setup, laplace_noise_behavior, Algorithms, Alg_Parameters, h0, PBar = PBar
    )

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(10 * np.log10(MC_measures["sKF_L_minorized"]["MSD"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_exact"]["MSD"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_integral"]["MSD"]))

plt.ylabel("MSD (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
h_L_num   = MC_measures["sKF_L_integral"]['h']
h_L_minorized = MC_measures["sKF_L_minorized"]['h']
h_L_exact = MC_measures["sKF_L_exact"]['h']

L = h_L_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_L_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_L_minorized[:, k], color=c, linestyle='--', linewidth=1, label=f"$w_{k}$ minorized")
  plt.plot(h_L_exact[:, k], color=c, linestyle='-.', label=f"$w_{k}$ exact")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF-L weights: numerical integration vs paper recursion")
# plt.legend(ncol=L, fontsize=9)
# --- PLOTTING THE PARAMETERS IN THE LEGEND ---
ax = plt.gca()
leg1 = ax.legend(ncol=L, fontsize=9, loc="upper right")
ax.add_artist(leg1)
def fmt(p):
    lgnd_str = "\n"
    for field in p._fields:
      value = getattr(p, field)
      lgnd_str += f"{field} -> {value}\n"
    return lgnd_str

ax.legend(handles=[],
          title=f"integral:\n{fmt(skf_L_int_params)}\n\nminorized and exact:\n{fmt(sKF_L_minorized_params)}",
          loc="lower right", fontsize=8, title_fontsize=8)
# --- ---
plt.grid(alpha=0.3)
plt.savefig("skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()